<a href="https://colab.research.google.com/github/DKavya8/chestxray-bias-audit/blob/main/notebooks/07_asbdc_vs_stdcal_paired_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np, pandas as pd, os, glob, json
from sklearn.linear_model import LogisticRegression
from scipy.stats import wilcoxon, ttest_rel

In [2]:
from google.colab import drive
drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/team-RACK-bias-paper"
RESULTS = f"{BASE}/results"
OUT = f"{RESULTS}/four_condition_grid"; os.makedirs(OUT, exist_ok=True)
print("BASE exists?", os.path.exists(BASE))

Mounted at /content/drive
BASE exists? True


In [3]:
!rm -rf /content/repo && git clone -q https://github.com/DKavya8/chestxray-bias-audit /content/repo
REPO = "/content/repo"
print("seed dirs:", len(glob.glob(f"{REPO}/splits/seed_*")), "(expect 10)")

seed dirs: 10 (expect 10)


In [4]:
scores = pd.read_parquet(f"{RESULTS}/densenet121_all_scores.parquet")
meta   = pd.read_csv(f"{RESULTS}/metadata_clean.csv")
print("scores:", scores.shape, "| meta:", meta.shape)

scores: (112120, 22) | meta: (112106, 24)


In [5]:
NIH_14 = ["Atelectasis","Consolidation","Infiltration","Pneumothorax","Edema","Emphysema",
          "Fibrosis","Effusion","Pneumonia","Pleural_Thickening","Cardiomegaly","Nodule","Mass","Hernia"]
sc = (scores[['Image Index'] + NIH_14]
      .rename(columns={f: f's_{f}' for f in NIH_14})
      .rename(columns={'Image Index': 'image_index'}))
md = (meta[['image_index','patient_id','age','sex','follow_up'] + NIH_14]
      .rename(columns={f: f'y_{f}' for f in NIH_14}))
img = md.merge(sc, on='image_index', how='inner', validate='one_to_one')
G = img.sort_values(['patient_id','follow_up']).drop_duplicates('patient_id', keep='first')
G = G.rename(columns={'patient_id': 'Patient ID'})
G['Patient ID'] = G['Patient ID'].astype(str)
G['age'] = G['age'].astype(float)
G['bin10'] = ((G['age']//10)*10).astype(int).astype(str)
print("G first scan per patient:", G.shape, "| sex:", dict(G['sex'].value_counts()))

G first scan per patient: (30797, 34) | sex: {'M': np.int64(16625), 'F': np.int64(14172)}


In [6]:
thr_by_seed = json.load(open(f"{REPO}/results/group_a_densenet/thresholds_by_seed.json"))
split_ids = {}
for d in sorted(glob.glob(f"{REPO}/splits/seed_*")):
    seed = d.split('seed_')[1]
    cal  = set(pd.read_csv(f"{d}/calibration_patients.csv")['Patient ID'].astype(str))
    test = set(pd.read_csv(f"{d}/test_patients.csv")['Patient ID'].astype(str))
    split_ids[seed] = (cal, test)
print("seeds:", len(split_ids))

seeds: 10


In [7]:
def pooled_fnr(df, thr, w=None):
    w = np.ones(len(df)) if w is None else np.asarray(w, float)
    fn = pos = 0.0
    for f in NIH_14:
        y = df[f'y_{f}'].to_numpy(); s = df[f's_{f}'].to_numpy(); p = (y == 1)
        pos += (w * p).sum(); fn += (w * (p & (s < thr[f]))).sum()
    return fn / pos if pos > 0 else np.nan

def sex_gap(df, thr, wcol=None):
    isF = df['sex'].to_numpy() == 'F'
    w = df[wcol].to_numpy() if wcol else np.ones(len(df))
    return pooled_fnr(df[isF], thr, w[isF]) - pooled_fnr(df[~isF], thr, w[~isF])

def ipw_weights(df):
    d = df.copy()
    fp = d[d.sex=='F']['bin10'].value_counts(normalize=True)
    mp = d[d.sex=='M']['bin10'].value_counts(normalize=True)
    bins = sorted(set(fp.index)|set(mp.index)); fp = fp.reindex(bins,fill_value=0); mp = mp.reindex(bins,fill_value=0)
    target = (fp+mp)/2
    d['w'] = d.apply(lambda r: target[r.bin10]/((fp if r.sex=='F' else mp)[r.bin10]), axis=1)
    return d

In [8]:
def _logit(p):
    p = np.clip(np.asarray(p,float), 1e-6, 1-1e-6); return np.log(p/(1-p))

def fit_calibrators(cal, use_age):
    amu, asd = cal['age'].mean(), (cal['age'].std() or 1.0)
    cals, fb = {}, {}
    for f in NIH_14:
        y = cal[f'y_{f}'].to_numpy().astype(int); s = cal[f's_{f}'].to_numpy()
        if y.sum() < 10 or np.unique(y).size < 2:
            cals[f] = None; fb[f] = int(y.sum()); continue
        X = _logit(s).reshape(-1,1)
        if use_age:
            z = (cal['age'].to_numpy()-amu)/asd; X = np.column_stack([_logit(s), z, z*z])
        cals[f] = (LogisticRegression(C=1e6, max_iter=2000).fit(X, y), amu, asd, use_age)
    return cals, fb

def apply_calibrators(df, cals):
    out = df.copy()
    for f in NIH_14:
        c = cals[f]
        if c is None: continue
        clf, amu, asd, use_age = c; s = df[f's_{f}'].to_numpy(); X = _logit(s).reshape(-1,1)
        if use_age:
            z = (df['age'].to_numpy()-amu)/asd; X = np.column_stack([_logit(s), z, z*z])
        out[f's_{f}'] = clf.predict_proba(X)[:,1]
    return out

In [9]:
rows = []
for seed, (cal_ids, test_ids) in split_ids.items():
    cal  = G[G['Patient ID'].isin(cal_ids)]
    test = G[G['Patient ID'].isin(test_ids)]
    thr  = thr_by_seed[seed]
    c3, _ = fit_calibrators(cal, use_age=False)   # standard calibration, age blind
    c4, _ = fit_calibrators(cal, use_age=True)    # ASBDC, age aware
    g3 = sex_gap(ipw_weights(apply_calibrators(test, c3)), thr, wcol='w')
    g4 = sex_gap(ipw_weights(apply_calibrators(test, c4)), thr, wcol='w')
    rows.append(dict(seed=seed, c3_ipw=g3, c4_ipw=g4, dS_ipw=g3-g4))
    print(f"{seed}:  std-cal={g3:+.4f}  ASBDC={g4:+.4f}  dS={g3-g4:+.4f}")
cond34 = pd.DataFrame(rows)
print("\nmean dS_ipw:", round(cond34['dS_ipw'].mean(), 4))

113462462:  std-cal=+0.0260  ASBDC=+0.0304  dS=-0.0045
1524358342:  std-cal=+0.0268  ASBDC=+0.0266  dS=+0.0002
1569714665:  std-cal=-0.0072  ASBDC=-0.0095  dS=+0.0023
1591287646:  std-cal=-0.0125  ASBDC=-0.0163  dS=+0.0038
2006902500:  std-cal=-0.0021  ASBDC=+0.0008  dS=-0.0029
2748406118:  std-cal=+0.0291  ASBDC=+0.0279  dS=+0.0012
2763601433:  std-cal=+0.0119  ASBDC=+0.0142  dS=-0.0024
342858866:  std-cal=-0.0024  ASBDC=+0.0090  dS=-0.0114
3658676649:  std-cal=+0.0044  ASBDC=+0.0063  dS=-0.0020
768519171:  std-cal=-0.0115  ASBDC=-0.0076  dS=-0.0039

mean dS_ipw: -0.0019


In [10]:
a = cond34['c3_ipw'].to_numpy()   # standard calibration gap
b = cond34['c4_ipw'].to_numpy()   # ASBDC gap
w_p = wilcoxon(a, b).pvalue
t_p = ttest_rel(a, b).pvalue
print(f"mean std-cal gap={a.mean():+.4f}  mean ASBDC gap={b.mean():+.4f}")
print(f"mean difference (std-cal minus ASBDC)={(a-b).mean():+.4f}")
print(f"paired Wilcoxon p={w_p:.4f}   paired t p={t_p:.4f}")

mean std-cal gap=+0.0063  mean ASBDC gap=+0.0082
mean difference (std-cal minus ASBDC)=-0.0019
paired Wilcoxon p=0.1934   paired t p=0.1884


In [11]:
B = 1000
rng = np.random.default_rng(20260823)
cache = {}
for seed, (cal_ids, test_ids) in split_ids.items():
    cal  = G[G['Patient ID'].isin(cal_ids)]
    test = G[G['Patient ID'].isin(test_ids)].reset_index(drop=True)
    c3, _ = fit_calibrators(cal, use_age=False)
    c4, _ = fit_calibrators(cal, use_age=True)
    t3 = apply_calibrators(test, c3).reset_index(drop=True)
    t4 = apply_calibrators(test, c4).reset_index(drop=True)
    cache[seed] = (t3, t4, t3.groupby('Patient ID').indices)

boot = np.empty(B)
for bi in range(B):
    diffs = []
    for seed in split_ids:
        t3, t4, idxmap = cache[seed]
        pids = np.array(list(idxmap.keys()))
        samp = rng.choice(pids, size=len(pids), replace=True)
        rows_ = np.concatenate([idxmap[p] for p in samp])
        g3 = sex_gap(ipw_weights(t3.iloc[rows_]), thr_by_seed[seed], wcol='w')
        g4 = sex_gap(ipw_weights(t4.iloc[rows_]), thr_by_seed[seed], wcol='w')
        diffs.append(g3 - g4)
    boot[bi] = np.mean(diffs)

lo, hi = np.percentile(boot, [2.5, 97.5]); mean_d = (a-b).mean(); beats = lo > 0
print(f"ASBDC minus standard-cal gap reduction: {mean_d:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
print("ASBDC beats age-blind calibration:", beats)
print("=>", "keep mitigation framing" if beats
      else "reframe to DECOMPOSITION / EVALUATION (ASBDC does not beat age-blind calibration)")

ASBDC minus standard-cal gap reduction: -0.0019  95% CI [-0.0051, +0.0011]
ASBDC beats age-blind calibration: False
=> reframe to DECOMPOSITION / EVALUATION (ASBDC does not beat age-blind calibration)


In [12]:
out = pd.DataFrame([{
    "comparison":"ASBDC_cond4_vs_standard_calibration_cond3", "standardization":"IPW",
    "metric":"residual_sex_fnr_gap_difference", "mean_diff":round(float(mean_d),6),
    "ci_lower":round(float(lo),6), "ci_upper":round(float(hi),6),
    "wilcoxon_p":round(float(w_p),6), "paired_t_p":round(float(t_p),6),
    "beats_age_blind":bool(beats), "n_splits":len(split_ids), "B":B}])
out.to_csv(f"{OUT}/asbdc_vs_stdcal_paired_test.csv", index=False)
print("wrote", f"{OUT}/asbdc_vs_stdcal_paired_test.csv"); out

wrote /content/drive/MyDrive/team-RACK-bias-paper/results/four_condition_grid/asbdc_vs_stdcal_paired_test.csv


,comparison,standardization,metric,mean_diff,ci_lower,ci_upper,wilcoxon_p,paired_t_p,beats_age_blind,n_splits,B
0,ASBDC_cond4_vs_standard_calibration_cond3,IPW,residual_sex_fnr_gap_difference,-0.001942,-0.005085,0.001134,0.193359,0.188423,False,10,1000
